# SRLG v4 Candidate Derivation (FREEZE-v4 shadow build)

**Project:** Provable Resilience Intelligence (Lakelink Fiber)  
**Tasks:** BUG-DP-069, BUG-QA-074 option b, split out by BUG-ML-V4-GUARD-SRLG-PIPELINE  
**Writes (shadow only):** `derived_srlg_shared_conduit_v4_candidate`, `derived_srlg_common_site_v4_candidate`, `derived_srlg_geo_proximity_v4_candidate`, `derived_srlg_shared_route_v4_candidate`, `derived_shared_risk_group_v4_candidate`, `unified_shared_risk_group_v4_candidate` (view), `path_diversity_conduit_v4_candidate` (cell 6, CG-JACCARD-v1, BUG-DP-070 / BUG-QA-085)  
**Reads:** `physical_resource_span_v4_candidate`, `physical_resource_conduit_v4_candidate`, `geographic_site`, `shared_risk_group`

This notebook never writes v3 FROZEN objects. The v3 pipeline stays in `srlg_colocation_derivation`.

* SPAN-* spans keep the v3 rules.
* BBS-* backbone segments join site and proximity SRLGs only at true terminals (seq=1 gives site_a, seq=segment_count gives site_z).
* shared_route groups parallel CDT-BBR conduits that have identical geom_wkt (31 routes).

**Write gate:** writes run only when the `write_enabled` widget is `true`. The default is `false`, which runs the read-only gates only.

BUG-QA-076: the geo_proximity bbox prefilter is cos(lat)-aware. Data rewritten in DE S42 (2026-10-26) via INSERT OVERWRITE (comments kept): 10 geo SRLGs, 1,342 rows; derived v4 = 42,171 rows.

BUG-QA-079: shared_conduit risk point = conduit site_a/site_z endpoint average (applied in DE S42). Note: CREATE OR REPLACE in cell 3 drops column comments; prefer INSERT OVERWRITE for targeted re-runs.

In [0]:
dbutils.widgets.dropdown("write_enabled", "false", ["false", "true"], "Write v4 candidate tables")
WRITE_ENABLED = dbutils.widgets.get("write_enabled") == "true"
S = "cdm_tmforum.provable_resilience_intelligence"
SP = f"{S}.physical_resource_span_v4_candidate"
CD = f"{S}.physical_resource_conduit_v4_candidate"
GS = f"{S}.geographic_site"

def run_write(sql_text: str):
    """Execute a shadow write only when WRITE_ENABLED. Refuses any non-v4 target."""
    head = sql_text.strip().split("\n", 1)[0]
    assert "_v4_candidate" in head, f"Refusing non-v4 write: {head}"
    if WRITE_ENABLED:
        spark.sql(sql_text)
        print("WROTE:", head[:120])
    else:
        print("SKIPPED (write_enabled=false):", head[:120])

print("WRITE_ENABLED =", WRITE_ENABLED)

In [0]:
spark.sql(f"""CREATE OR REPLACE TEMP VIEW span_site_term AS
SELECT span_id, site_a_id AS site_id FROM {SP} WHERE is_active AND span_id NOT LIKE 'BBS-%'
UNION SELECT span_id, site_z_id FROM {SP} WHERE is_active AND span_id NOT LIKE 'BBS-%'
UNION SELECT span_id, site_a_id FROM {SP} WHERE is_active AND span_id LIKE 'BBS-%' AND segment_seq=1
UNION SELECT span_id, site_z_id FROM {SP} WHERE is_active AND span_id LIKE 'BBS-%' AND segment_seq=segment_count""")

run_write(f"""CREATE OR REPLACE TABLE {S}.derived_srlg_shared_conduit_v4_candidate AS
WITH m AS (
  SELECT sp.span_id, sp.conduit_id,
    CASE WHEN sp.span_id LIKE 'BBS-%' THEN st_y(st_centroid(st_geomfromwkt(sp.geom_wkt))) ELSE (a.latitude+z.latitude)/2 END lat,
    CASE WHEN sp.span_id LIKE 'BBS-%' THEN st_x(st_centroid(st_geomfromwkt(sp.geom_wkt))) ELSE (a.longitude+z.longitude)/2 END lon
  FROM {SP} sp JOIN {GS} a ON sp.site_a_id=a.site_id JOIN {GS} z ON sp.site_z_id=z.site_id
  WHERE sp.is_active AND sp.conduit_id IS NOT NULL),
g AS (SELECT conduit_id, count(*) member_count FROM m GROUP BY conduit_id HAVING count(*)>=2),
-- BUG-QA-079: risk point = average of the conduit's site_a/site_z coordinates (endpoint-average convention, matches shared_route centroid on CDT-BBR)
e AS (SELECT c.conduit_id, (a.latitude+z.latitude)/2 rlat, (a.longitude+z.longitude)/2 rlon
      FROM {CD} c JOIN {GS} a ON c.site_a_id=a.site_id JOIN {GS} z ON c.site_z_id=z.site_id)
SELECT concat('SRLG-COND-',m.conduit_id) srlg_id, concat('Shared Conduit ',m.conduit_id) srlg_name, 'shared_conduit' srlg_type,
  m.span_id, m.conduit_id, CAST(NULL AS STRING) site_id, e.rlat risk_latitude, e.rlon risk_longitude, 'critical' severity,
  g.member_count, current_timestamp() derived_at, 'DERIVED' origin, 'v4_candidate' derivation_version
FROM m JOIN g USING (conduit_id) JOIN e USING (conduit_id)""")

run_write(f"""CREATE OR REPLACE TABLE {S}.derived_srlg_common_site_v4_candidate AS
WITH g AS (SELECT site_id, count(DISTINCT span_id) member_count FROM span_site_term WHERE site_id IS NOT NULL GROUP BY site_id HAVING count(DISTINCT span_id)>=2)
SELECT concat('SRLG-SITE-',t.site_id) srlg_id, concat('Site Colocation ',gs.site_name) srlg_name, 'central_office' srlg_type,
  t.span_id, CAST(NULL AS STRING) conduit_id, t.site_id, gs.latitude risk_latitude, gs.longitude risk_longitude, 'high' severity,
  CAST(g.member_count AS BIGINT) member_count, current_timestamp() derived_at, 'DERIVED' origin, 'v4_candidate' derivation_version
FROM span_site_term t JOIN g USING (site_id) JOIN {GS} gs ON t.site_id=gs.site_id""")

run_write(f"""CREATE OR REPLACE TABLE {S}.derived_srlg_geo_proximity_v4_candidate AS
WITH sp AS (
  SELECT s1.site_id s1, s2.site_id s2,
    6371000*2*ASIN(SQRT(POWER(SIN(RADIANS(s2.latitude-s1.latitude)/2),2)+COS(RADIANS(s1.latitude))*COS(RADIANS(s2.latitude))*POWER(SIN(RADIANS(s2.longitude-s1.longitude)/2),2))) d,
    (s1.latitude+s2.latitude)/2 mlat, (s1.longitude+s2.longitude)/2 mlon
  -- BUG-QA-076: cos(lat)-aware bbox prefilter (0.0046 deg ~ 511 m of latitude; the dlon bound is scaled by 1/cos(mean lat)).
  -- Validated: 680/680 site pairs <=500 m recalled vs a full cross join (the old fixed 0.005 deg box found 623).
  FROM {GS} s1 JOIN {GS} s2 ON s1.site_id<s2.site_id
    AND ABS(s1.latitude-s2.latitude)<0.0046
    AND ABS(s1.longitude-s2.longitude)<0.0046/COS(RADIANS((s1.latitude+s2.latitude)/2))
  WHERE s1.is_active AND s2.is_active),
cp AS (SELECT * FROM sp WHERE d<=500),
mem AS (SELECT DISTINCT cp.*, t.span_id FROM cp JOIN span_site_term t ON t.site_id IN (cp.s1, cp.s2)),
g AS (SELECT s1,s2,count(DISTINCT span_id) n FROM mem GROUP BY s1,s2 HAVING count(DISTINCT span_id)>=2)
SELECT concat('SRLG-GEO-',m.s1,'-',m.s2) srlg_id, concat('Proximity ',m.s1,' / ',m.s2,' (',round(m.d,1),'m)') srlg_name, 'geographic_proximity' srlg_type,
  m.span_id, CAST(NULL AS STRING) conduit_id, m.s1 site_id, m.mlat risk_latitude, m.mlon risk_longitude, 'medium' severity,
  CAST(g.n AS BIGINT) member_count, current_timestamp() derived_at, 'DERIVED' origin, 'v4_candidate' derivation_version
FROM mem m JOIN g USING (s1,s2)""")

run_write(f"""CREATE OR REPLACE TABLE {S}.derived_srlg_shared_route_v4_candidate AS
WITH c AS (SELECT conduit_id, geom_wkt, least(site_a_id,site_z_id) sx, greatest(site_a_id,site_z_id) sy FROM {CD} WHERE is_active AND conduit_id LIKE 'CDT-BBR-%'),
grp AS (SELECT geom_wkt, min(sx) sx, min(sy) sy, count(*) n_conduits FROM c GROUP BY geom_wkt HAVING count(*)>=2),
m AS (SELECT grp.sx, grp.sy, grp.n_conduits, sp.span_id, sp.conduit_id,
        st_y(st_centroid(st_geomfromwkt(grp.geom_wkt))) rlat, st_x(st_centroid(st_geomfromwkt(grp.geom_wkt))) rlon
      FROM grp JOIN c ON c.geom_wkt=grp.geom_wkt JOIN {SP} sp ON sp.conduit_id=c.conduit_id AND sp.is_active)
SELECT concat('SRLG-ROUTE-',sx,'-',sy) srlg_id, concat('Shared Backbone Route ',sx,' - ',sy,' (',n_conduits,' parallel conduits)') srlg_name,
  'shared_route' srlg_type, span_id, conduit_id, CAST(NULL AS STRING) site_id, rlat risk_latitude, rlon risk_longitude, 'critical' severity,
  count(*) OVER (PARTITION BY sx,sy) member_count, current_timestamp() derived_at, 'DERIVED' origin, 'v4_candidate' derivation_version
FROM m""")

cols = "srlg_id, srlg_name, srlg_type, span_id, conduit_id, site_id, risk_latitude, risk_longitude, severity, member_count, derived_at, origin, derivation_version"
run_write(f"""CREATE OR REPLACE TABLE {S}.derived_shared_risk_group_v4_candidate AS
SELECT {cols} FROM {S}.derived_srlg_shared_conduit_v4_candidate UNION ALL
SELECT {cols} FROM {S}.derived_srlg_common_site_v4_candidate UNION ALL
SELECT {cols} FROM {S}.derived_srlg_geo_proximity_v4_candidate UNION ALL
SELECT {cols} FROM {S}.derived_srlg_shared_route_v4_candidate""")
# CREATE OR REPLACE drops column comments. Re-apply them from the comment snapshot after any re-run (DEC-PRI-FREEZE-COMMENTS).

In [0]:
run_write(f"""CREATE OR REPLACE VIEW {S}.unified_shared_risk_group_v4_candidate (
  srlg_id COMMENT 'Unique SRLG identifier',
  srlg_name COMMENT 'Descriptive name of the shared risk group',
  srlg_type COMMENT 'Risk type: bridge | rail_crossing | shared_conduit | central_office | tunnel | power_feed | geographic_proximity | shared_route',
  span_id COMMENT 'FK to physical_resource_span_v4_candidate.span_id (member span; SPAN-* or BBS-*)',
  conduit_id COMMENT 'FK to physical_resource_conduit_v4_candidate.conduit_id if conduit/route-based',
  site_id COMMENT 'FK to geographic_site.site_id if site-based (CO/DC risk)',
  risk_latitude COMMENT 'Latitude (WGS84) of the shared-risk point',
  risk_longitude COMMENT 'Longitude (WGS84) of the shared-risk point',
  severity COMMENT 'Risk severity: critical | high | medium | low',
  source COMMENT 'SRLG origin: seed | derived_shared_conduit | derived_common_site | derived_geo_proximity | derived_shared_route')
COMMENT 'Lakelink Fiber unified SRLG memberships, FREEZE-v4 CANDIDATE (BUG-DP-069 + BUG-QA-074). Seed shared_risk_group (unchanged) + v4 derived SRLGs computed on physical_resource_span_v4_candidate, including BBS-* backbone segments and a shared_route SRLG for parallel CDT-BBR conduits on identical geometry. One row per (srlg_id, span_id, source). Shadow build: do not consume until cutover.'
AS SELECT srlg_id,srlg_name,srlg_type,span_id,conduit_id,site_id,risk_latitude,risk_longitude,severity,'seed' FROM {S}.shared_risk_group
UNION ALL SELECT srlg_id,srlg_name,srlg_type,span_id,conduit_id,site_id,risk_latitude,risk_longitude,severity,
  concat('derived_', CASE srlg_type WHEN 'central_office' THEN 'common_site' WHEN 'geographic_proximity' THEN 'geo_proximity' ELSE srlg_type END)
FROM {S}.derived_shared_risk_group_v4_candidate""")

In [0]:
g = spark.sql(f"""
WITH d AS (SELECT * FROM {S}.derived_shared_risk_group_v4_candidate),
multi AS (SELECT conduit_id FROM {SP} WHERE is_active AND conduit_id IS NOT NULL GROUP BY conduit_id HAVING count(*)>=2)
SELECT
  (SELECT count(*) FROM d) AS derived_rows,
  (SELECT count(*) FROM {S}.unified_shared_risk_group_v4_candidate) AS unified_rows,
  (SELECT count(DISTINCT srlg_id) FROM {S}.unified_shared_risk_group_v4_candidate) AS unified_srlgs,
  (SELECT count(*) FROM d LEFT ANTI JOIN {SP} s USING (span_id)) AS g1_span_fk_orphans,
  (SELECT count(*) FROM d WHERE conduit_id IS NOT NULL AND conduit_id NOT IN (SELECT conduit_id FROM {CD})) AS g2_conduit_fk_orphans,
  (SELECT count(*) FROM multi WHERE conduit_id NOT IN (SELECT conduit_id FROM d WHERE srlg_type='shared_conduit')) AS g3_uncovered_multi_conduits,
  (SELECT count(DISTINCT srlg_id) FROM d WHERE srlg_type='shared_route') AS g4_shared_route_srlgs,
  (SELECT count(*) FROM d WHERE span_id LIKE 'BBS-%' AND srlg_type IN ('central_office','geographic_proximity')
     AND span_id NOT IN (SELECT span_id FROM {SP} WHERE segment_seq=1 OR segment_seq=segment_count)) AS g5_bbs_nonterminal_site_members
""")
display(g)
r = g.collect()[0]
assert r.g1_span_fk_orphans == 0 and r.g2_conduit_fk_orphans == 0 and r.g3_uncovered_multi_conduits == 0 and r.g5_bbs_nonterminal_site_members == 0, r
print("All gates passed")

In [0]:
# BUG-DP-070 / BUG-QA-085: writer for path_diversity_conduit_v4_candidate (formula CG-JACCARD-v1).
# Hop -> v4 conduit: SPAN-* via span_v4.conduit_id; legacy BB-<route> hops -> CDT-BBR-<route>.
# Risk groups per path = {conduit_id} U {SRLG-ROUTE-* shared_route SRLGs of those conduits}.
# conduit_grain_diversity_score = 100*(1 - |W n P| / |W u P|), 1 dp (NULL if no protect path).
# path_diversity_score_v4 = LEAST(v3 pinned score, conduit_grain score); v3 value when no protect path.
# v3 pin = resilience_score_snapshot VERSION AS OF 22 (immutable). Shadow write only; default dry run.
PDC = f"{S}.path_diversity_conduit_v4_candidate"

PDC_SELECT = f"""
WITH hop AS (
  SELECT p.service_id, p.path_type, p.span_id,
         CASE WHEN p.span_id LIKE 'BB-%' THEN concat('CDT-BBR-', substr(p.span_id, 4)) ELSE s.conduit_id END AS conduit_id
  FROM {S}.service_path p
  LEFT JOIN {SP} s ON p.span_id = s.span_id AND p.span_id NOT LIKE 'BB-%'),
route AS (SELECT DISTINCT conduit_id, srlg_id FROM {S}.unified_shared_risk_group_v4_candidate WHERE srlg_type = 'shared_route'),
grp AS (
  SELECT DISTINCT service_id, path_type, conduit_id AS group_id, 'conduit' AS gkind FROM hop WHERE conduit_id IS NOT NULL
  UNION
  SELECT DISTINCT h.service_id, h.path_type, r.srlg_id, 'route' FROM hop h JOIN route r ON h.conduit_id = r.conduit_id),
gp AS (
  SELECT service_id, group_id, gkind,
         max(CASE WHEN path_type='working' THEN 1 ELSE 0 END) w, max(CASE WHEN path_type='protect' THEN 1 ELSE 0 END) p
  FROM grp GROUP BY service_id, group_id, gkind),
ga AS (
  SELECT service_id,
    count(CASE WHEN gkind='conduit' AND w=1 THEN 1 END) working_conduit_count,
    count(CASE WHEN gkind='conduit' AND p=1 THEN 1 END) protect_conduit_count,
    count(CASE WHEN gkind='conduit' AND w=1 AND p=1 THEN 1 END) shared_conduit_count,
    count(CASE WHEN gkind='route' AND w=1 AND p=1 THEN 1 END) shared_route_srlg_count,
    count(*) union_group_count,
    count(CASE WHEN w=1 AND p=1 THEN 1 END) shared_group_count,
    array_sort(collect_list(CASE WHEN gkind='conduit' AND w=1 AND p=1 THEN group_id END)) shared_conduit_ids
  FROM gp GROUP BY service_id),
sp AS (SELECT service_id, max(CASE WHEN path_type='protect' THEN 1 ELSE 0 END)=1 has_protect_path FROM hop GROUP BY service_id),
ss AS (
  SELECT w.service_id, count(DISTINCT w.span_id) shared_span_count
  FROM hop w JOIN hop p ON w.service_id=p.service_id AND w.span_id=p.span_id AND w.path_type='working' AND p.path_type='protect'
  GROUP BY w.service_id),
v3 AS (SELECT service_id, path_diversity_score v3_path_diversity_score FROM {S}.resilience_score_snapshot VERSION AS OF 22),
b AS (
  SELECT si.service_id, si.sold_as_diverse, coalesce(sp.has_protect_path,false) has_protect_path,
    coalesce(ga.working_conduit_count,0) working_conduit_count, coalesce(ga.protect_conduit_count,0) protect_conduit_count,
    coalesce(ga.shared_conduit_count,0) shared_conduit_count, coalesce(ga.shared_route_srlg_count,0) shared_route_srlg_count,
    coalesce(ss.shared_span_count,0) shared_span_count, coalesce(ga.union_group_count,0) union_group_count,
    coalesce(ga.shared_group_count,0) shared_group_count, coalesce(ga.shared_conduit_ids, array()) shared_conduit_ids,
    v3.v3_path_diversity_score
  FROM {S}.service_inventory si
  LEFT JOIN ga ON si.service_id=ga.service_id LEFT JOIN sp ON si.service_id=sp.service_id
  LEFT JOIN ss ON si.service_id=ss.service_id LEFT JOIN v3 ON si.service_id=v3.service_id),
c AS (
  SELECT *, CASE WHEN has_protect_path AND union_group_count>0
                 THEN round(100*(1 - shared_group_count/union_group_count), 1) END conduit_grain_diversity_score
  FROM b)
SELECT service_id, sold_as_diverse, has_protect_path, working_conduit_count, protect_conduit_count, shared_conduit_count,
  shared_route_srlg_count, shared_span_count, union_group_count, shared_group_count, shared_conduit_ids,
  conduit_grain_diversity_score, v3_path_diversity_score,
  CASE WHEN conduit_grain_diversity_score IS NULL THEN v3_path_diversity_score
       ELSE least(v3_path_diversity_score, conduit_grain_diversity_score) END path_diversity_score_v4,
  CASE WHEN NOT has_protect_path THEN 'no_protect_path'
       WHEN shared_span_count>0 THEN 'shared_span'
       WHEN shared_conduit_count>0 THEN 'shared_conduit'
       WHEN shared_route_srlg_count>0 THEN 'shared_route_only'
       ELSE 'conduit_diverse' END conduit_diversity_verdict,
  'CG-JACCARD-v1' formula_version, 'DERIVED' origin, current_timestamp() created_at
FROM c"""
spark.sql(f"CREATE OR REPLACE TEMP VIEW pdc_v4_dry AS {PDC_SELECT}")

# DDL with comments. CREATE TABLE IF NOT EXISTS + INSERT OVERWRITE keeps comments (never CREATE OR REPLACE).
run_write(f"""CREATE TABLE IF NOT EXISTS {S}.path_diversity_conduit_v4_candidate (
  service_id STRING COMMENT 'FK to service_inventory.service_id. One row per Lakelink Fiber service (8,000).',
  sold_as_diverse BOOLEAN COMMENT 'From service_inventory: service sold with a path diversity guarantee.',
  has_protect_path BOOLEAN COMMENT 'True if service_path has at least one protect hop for the service.',
  working_conduit_count BIGINT COMMENT 'Distinct v4 conduits on the working path (SPAN-* via span_v4.conduit_id; BB-<route> hops map to CDT-BBR-<route>).',
  protect_conduit_count BIGINT COMMENT 'Distinct v4 conduits on the protect path, same mapping as working_conduit_count.',
  shared_conduit_count BIGINT COMMENT 'Distinct conduits on both working and protect paths (physical single point of failure).',
  shared_route_srlg_count BIGINT COMMENT 'Distinct shared_route SRLGs (parallel CDT-BBR conduits on identical geometry, SRLG-ROUTE-*) on both paths.',
  shared_span_count BIGINT COMMENT 'Distinct span_ids present on both working and protect paths (span overlap).',
  union_group_count BIGINT COMMENT 'Size of the union of conduit-grain risk groups (conduits plus shared_route SRLGs) across working and protect.',
  shared_group_count BIGINT COMMENT 'Size of the intersection of conduit-grain risk groups between working and protect = shared_conduit_count + shared_route_srlg_count.',
  shared_conduit_ids ARRAY<STRING> COMMENT 'Sorted list of conduit_ids shared by the working and protect paths.',
  conduit_grain_diversity_score DOUBLE COMMENT 'Conduit-grain diversity 0-100 = 100*(1 - shared_group_count/union_group_count), 1 dp. NULL when there is no protect path.',
  v3_path_diversity_score DOUBLE COMMENT 'SRLG-grain path_diversity_score pinned from v3 snapshot v22 (carried in resilience_score_snapshot_v4_candidate).',
  path_diversity_score_v4 DOUBLE COMMENT 'Official v4 path diversity (0-100) = LEAST(v3_path_diversity_score, conduit_grain_diversity_score); v3 value when no protect path.',
  conduit_diversity_verdict STRING COMMENT 'no_protect_path | conduit_diverse (no shared group) | shared_span | shared_conduit (span-disjoint) | shared_route_only.',
  formula_version STRING COMMENT 'Formula identifier: CG-JACCARD-v1 (BUG-DP-070).',
  origin STRING COMMENT 'Data provenance per P14-DP standard. DERIVED = computed from other tables.',
  created_at TIMESTAMP COMMENT 'UTC timestamp when this shadow row was written.')
COMMENT 'Lakelink Fiber conduit-grain path diversity per service, FREEZE-v4 CANDIDATE (BUG-DP-070, shadow build, not live). Detects sold-diverse services whose working and protect paths share a conduit or a parallel shared_route corridor, which the SRLG-grain v3 score misses. Reads service_path, physical_resource_span_v4_candidate, unified_shared_risk_group_v4_candidate, resilience_score_snapshot_v4_candidate. Writer: pipelines/srlg_v4_candidate_derivation.'""")

run_write(f"""INSERT OVERWRITE {S}.path_diversity_conduit_v4_candidate
SELECT * FROM pdc_v4_dry""")

# Dry-run reproducibility check (always runs): recompute vs live table, excluding created_at.
if spark.catalog.tableExists(PDC):
    _cols = ",".join(c for c in spark.table(PDC).columns if c != "created_at")
    _d1 = spark.sql(f"SELECT {_cols} FROM pdc_v4_dry EXCEPT ALL SELECT {_cols} FROM {PDC}").count()
    _d2 = spark.sql(f"SELECT {_cols} FROM {PDC} EXCEPT ALL SELECT {_cols} FROM pdc_v4_dry").count()
    print(f"Recompute vs table diffs: dry-table={_d1}, table-dry={_d2}")

# Gates G6-G8 on the recompute (dry run) or on the written table (write_enabled).
_src = PDC if WRITE_ENABLED else "pdc_v4_dry"
g2 = spark.sql(f"""SELECT
  (SELECT count(*) FROM {_src}) AS total_rows,
  (SELECT count(DISTINCT service_id) FROM {_src}) AS g6_unique_services,
  (SELECT count(*) FROM {_src} WHERE path_diversity_score_v4 > v3_path_diversity_score) AS g7_v4_gt_v3,
  (SELECT count(*) FROM {_src} WHERE path_diversity_score_v4 IS NULL OR path_diversity_score_v4 NOT BETWEEN 0 AND 100
     OR (conduit_grain_diversity_score IS NOT NULL AND conduit_grain_diversity_score NOT BETWEEN 0 AND 100)
     OR (has_protect_path AND conduit_grain_diversity_score IS NULL)) AS g8_null_or_out_of_range,
  (SELECT count(*) FROM {S}.service_inventory) AS expected_services""")
display(g2)
r2 = g2.collect()[0]
assert r2.total_rows == r2.g6_unique_services == r2.expected_services, f"G6 failed: {r2}"
assert r2.g7_v4_gt_v3 == 0, f"G7 failed: {r2}"
assert r2.g8_null_or_out_of_range == 0, f"G8 failed: {r2}"
print("Gates G6-G8 passed on", _src)